# GPU Inference Review (CPU-only notebook)

This notebook is **CPU-only** — it never imports `torch` and never loads the model. The magtrain Jupyter kernel (`cns_vnv`) has no GPU access, so batch inference is run separately on a GPU compute node via:

```bash
sbatch scripts/predict.slurm
```

That job (`scripts/predict.py`) runs sliding-window inference with the trained checkpoint and writes CPU-viewable artifacts to `experiments/predictions/` (or wherever `output.output_dir` / `--output-dir` points):

- `predictions/*_pred.nii.gz` — predicted segmentation masks (NIfTI)
- `overlays/*_overlay.png` — 4-panel mid-slice overlay images (input / GT / prediction / GT+pred overlay)
- `dice_per_subject.csv` — per-subject Dice / HD95 / volume error / surface Dice
- `metrics_summary.yaml` — aggregate metrics (overall + per-site)

This notebook loads and visualizes those artifacts only.

In [ ]:
from pathlib import Path

import matplotlib.image as mpimg
import matplotlib.pyplot as plt
import pandas as pd
import yaml

%matplotlib inline

PROJECT_ROOT = Path.cwd().parent
OUTPUT_DIR = PROJECT_ROOT / "experiments" / "predictions"  # override if predict.slurm used --output-dir
PREDICTIONS_DIR = OUTPUT_DIR / "predictions"
OVERLAYS_DIR = OUTPUT_DIR / "overlays"
TARGET_DICE = 0.93  # Phase 1 target

print(f"Output dir:  {OUTPUT_DIR}")
print(f"Exists:      {OUTPUT_DIR.exists()}")

## Load per-subject metrics and aggregate summary

In [ ]:
dice_csv = OUTPUT_DIR / "dice_per_subject.csv"
summary_yaml = OUTPUT_DIR / "metrics_summary.yaml"

if not dice_csv.exists() or not summary_yaml.exists():
    raise FileNotFoundError(
        f"Inference artifacts not found under {OUTPUT_DIR}.\n"
        "Run the GPU inference job first: sbatch scripts/predict.slurm"
    )

df = pd.read_csv(dice_csv)
with open(summary_yaml) as f:
    summary = yaml.safe_load(f)

print(f"{len(df)} subjects loaded")
df.head()

In [ ]:
overall = pd.DataFrame(summary["overall"]).T
overall

In [ ]:
dice_mean = summary["overall"]["dice"]["mean"]
status = "MEETS" if dice_mean >= TARGET_DICE else "BELOW"
print(f"Mean Dice: {dice_mean:.4f}  vs.  Phase 1 target {TARGET_DICE:.2f}  ->  {status} target")
print(f"n_subjects: {summary['n_subjects']}")

## Per-site breakdown

In [ ]:
per_site_rows = {
    site: {
        "dice_mean": stats["dice"]["mean"],
        "dice_std": stats["dice"]["std"],
        "n": stats["dice"]["n"],
    }
    for site, stats in summary["per_site"].items()
}
per_site_df = pd.DataFrame(per_site_rows).T.sort_values("dice_mean", ascending=False)
per_site_df

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
per_site_df["dice_mean"].plot.bar(yerr=per_site_df["dice_std"], ax=ax, color="steelblue", capsize=4)
ax.axhline(TARGET_DICE, color="red", linestyle="--", label=f"Phase 1 target ({TARGET_DICE})")
ax.set_ylabel("Dice")
ax.set_title("Mean Dice by site")
ax.legend()
plt.tight_layout()
plt.show()

## Distribution of per-subject Dice

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(df["dice"], bins=20, color="steelblue", edgecolor="black")
ax.axvline(TARGET_DICE, color="red", linestyle="--", label=f"Phase 1 target ({TARGET_DICE})")
ax.axvline(df["dice"].mean(), color="green", linestyle="-", label=f"Mean ({df['dice'].mean():.3f})")
ax.set_xlabel("Dice")
ax.set_ylabel("Subjects")
ax.set_title("Per-subject Dice distribution")
ax.legend()
plt.tight_layout()
plt.show()

## Leaderboard: best / worst subjects

In [ ]:
N = 5
print("--- Best subjects ---")
display(df.sort_values("dice", ascending=False).head(N))
print("--- Worst subjects ---")
display(df.sort_values("dice", ascending=True).head(N))

## Overlay viewer

Shows the 4-panel mid-slice overlay PNG (Input MRI / Ground Truth / Prediction / Green=GT,Red=Pred) written by `scripts/predict.py` for a given subject.

In [ ]:
def show_overlay(subject_id: str) -> None:
    """Display the saved overlay PNG for a subject."""
    path = OVERLAYS_DIR / f"{subject_id}_overlay.png"
    if not path.exists():
        print(f"No overlay found for {subject_id} at {path}")
        return
    img = mpimg.imread(path)
    fig, ax = plt.subplots(figsize=(16, 4))
    ax.imshow(img)
    ax.axis("off")
    plt.show()

In [ ]:
print("Worst subjects (most useful for debugging):")
for subject_id in df.sort_values("dice", ascending=True).head(3)["subject"]:
    show_overlay(subject_id)

In [ ]:
print("Best subjects:")
for subject_id in df.sort_values("dice", ascending=False).head(3)["subject"]:
    show_overlay(subject_id)

In [ ]:
# Look up any single subject by id
# show_overlay("sub-XXXXX")